In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# ====== CONFIG ======
dirs = {
    "Proteomic Ratios": "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE",
    "Raw Proteins + Demographics": "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_fixed",
    "Demographics Only Baseline": "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_BASELINE_demo_only",
    "Stratified Random Baseline": "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_BASELINE_random",
}

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]

def safe_cls(c):
    return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

rows = []
for model_name, model_dir in dirs.items():
    for cls in classes:
        aucs, aps, f1s = [], [], []
        for seed in seeds:
            fp = os.path.join(model_dir, f"seed{seed}_{safe_cls(cls)}.csv")
            if not os.path.exists(fp):
                print(f"  MISSING: {fp}")
                continue
            df = pd.read_csv(fp)
            aucs.append(roc_auc_score(df["y_true"], df["y_score"]))
            aps.append(average_precision_score(df["y_true"], df["y_score"]))
            y_pred = (df["y_score"] >= 0.5).astype(int)
            f1s.append(f1_score(df["y_true"], y_pred, zero_division=0))
        if aucs:
            rows.append({
                "Model": model_name,
                "Class": cls,
                "AUC": f"{np.mean(aucs):.4f} ± {np.std(aucs):.4f}",
                "AP": f"{np.mean(aps):.4f} ± {np.std(aps):.4f}",
                "F1": f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
            })

result = pd.DataFrame(rows)
print(result.to_string(index=False))

                      Model Class             AUC              AP              F1
           Proteomic Ratios   MCI 0.6457 ± 0.0152 0.4348 ± 0.0245 0.3296 ± 0.0256
           Proteomic Ratios   NCI 0.7469 ± 0.0287 0.7398 ± 0.0348 0.7138 ± 0.0157
           Proteomic Ratios    AD 0.8407 ± 0.0169 0.5887 ± 0.0220 0.4603 ± 0.0426
           Proteomic Ratios   AD+ 0.8136 ± 0.0765 0.4429 ± 0.1850 0.1758 ± 0.2255
Raw Proteins + Demographics   MCI 0.5176 ± 0.0240 0.3011 ± 0.0231 0.0707 ± 0.0934
Raw Proteins + Demographics   NCI 0.6749 ± 0.0414 0.6670 ± 0.0443 0.6689 ± 0.0308
Raw Proteins + Demographics    AD 0.7161 ± 0.0180 0.4102 ± 0.0311 0.1982 ± 0.0553
Raw Proteins + Demographics   AD+ 0.6288 ± 0.1325 0.0531 ± 0.0344 0.0000 ± 0.0000
 Demographics Only Baseline   MCI 0.5508 ± 0.0063 0.3349 ± 0.0129 0.0442 ± 0.0765
 Demographics Only Baseline   NCI 0.6447 ± 0.0244 0.6555 ± 0.0289 0.6665 ± 0.0302
 Demographics Only Baseline    AD 0.6137 ± 0.0367 0.2592 ± 0.0401 0.0080 ± 0.0160
 Demographics On